# Training

---

### Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

/opt/anaconda3/envs/fraud_detect_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

### Loading dataset

In [2]:
df = pd.read_csv("../data/data.csv")

In [3]:
df["event_time"] = pd.to_datetime(df["event_time"], format="ISO8601")
df = df.sort_values(["account_id", "event_time"]).reset_index(drop=True)

In [4]:
print(df.shape)

(209102, 6)


In [5]:
df.head()

,account_id,amount,event_time,is_home,is_domestic,is_fraud
0,ML_00000,815000,2026-06-24 12:52:15.521757+00:00,1,1,False
1,ML_00000,838000,2026-06-24 13:27:06.843143+00:00,1,1,False
2,ML_00000,974000,2026-06-24 18:54:56.766292+00:00,1,1,False
3,ML_00000,1036000,2026-06-25 10:56:52.600864+00:00,1,1,False
4,ML_00002,24558000,2026-06-24 12:53:20.284844+00:00,1,1,False


---

### Feature Engineering

In [6]:
results = []
for _, grp in df.groupby("account_id"):
    grp = grp.sort_values("event_time").set_index("event_time")
    grp["avg_amount_24h"]         = grp["amount"].rolling("24h", closed="left", min_periods=1).mean()
    grp["tx_count_1h"]            = grp["amount"].rolling("1h",  closed="left", min_periods=0).count()
    grp["tx_count_3h"]            = grp["amount"].rolling("3h",  closed="left", min_periods=0).count()
    grp["time_since_last_tx_sec"] = grp.index.to_series().diff().dt.total_seconds()
    results.append(grp.reset_index())

df = pd.concat(results, ignore_index=True)
df["hour_of_day"] = df["event_time"].dt.hour

---

### Cleaning data

In [7]:
df["avg_amount_24h"]         = df["avg_amount_24h"].fillna(df["amount"])
df["log_amount_ratio_24h"]   = np.log1p(df["amount"] / df["avg_amount_24h"])
df["time_since_last_tx_sec"] = df["time_since_last_tx_sec"].fillna(0.0)

In [8]:
df["is_home"]     = df["is_home"].astype(int)
df["is_domestic"] = df["is_domestic"].astype(int)

---

### Spliting train/ test dataset

In [9]:
fraud_accounts  = df[df["is_fraud"]]["account_id"].unique()
normal_accounts = df[~df["is_fraud"]]["account_id"].unique()

In [10]:
rng = np.random.default_rng(42)
fraud_accounts  = np.array(fraud_accounts)
normal_accounts = np.array(normal_accounts)
rng.shuffle(fraud_accounts)
rng.shuffle(normal_accounts)

In [11]:
n_fraud_train  = int(len(fraud_accounts)  * 0.8)
n_normal_train = int(len(normal_accounts) * 0.8)

In [12]:
train_accounts = set(np.concatenate([fraud_accounts[:n_fraud_train],
                                     normal_accounts[:n_normal_train]]))
test_accounts  = set(np.concatenate([fraud_accounts[n_fraud_train:],
                                     normal_accounts[n_normal_train:]]))

In [13]:
FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "log_amount_ratio_24h",
    "tx_count_1h", "tx_count_3h",
    "time_since_last_tx_sec",
]

In [14]:
train = df[df["account_id"].isin(train_accounts)]
test  = df[df["account_id"].isin(test_accounts)]

X_train, y_train = train[FEATURES], train["is_fraud"].astype(int)
X_test,  y_test  = test[FEATURES],  test["is_fraud"].astype(int)

In [15]:
print(f"Train: {len(train):,} rows  |  fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test : {len(test):,} rows   |  fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

Train: 168,970 rows  |  fraud: 4,504 (2.67%)
Test : 43,455 rows   |  fraud: 1,687 (3.88%)


---

### Training model

In [16]:
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())

In [17]:
def objective(trial):
    params = {
        "n_estimators"         : 300,
        "max_depth"            : trial.suggest_int("max_depth", 3, 5),  
        "learning_rate"        : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"            : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree"     : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight"     : trial.suggest_int("min_child_weight", 3, 15), 
        "scale_pos_weight"     : scale_pos_weight,
        "eval_metric"          : "aucpr",
        "early_stopping_rounds": 20,
        "random_state"         : 42,
        "reg_alpha"            : trial.suggest_float("reg_alpha", 0.1, 2.0),   # L1
        "reg_lambda"           : trial.suggest_float("reg_lambda", 1.0, 5.0),  # L2
    }
    m = XGBClassifier(**params)
    m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_prob = m.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, y_prob)

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

Best trial: 12. Best value: 0.999902: 100%|██████████| 20/20 [00:12<00:00,  1.61it/s]


In [19]:
best = study.best_params
best.update({
    "n_estimators"         : 500,
    "scale_pos_weight"     : scale_pos_weight,
    "eval_metric"          : "aucpr",
    "early_stopping_rounds": 20,
    "random_state"         : 42,
    "monotone_constraints" : {"log_amount_ratio_24h": 1},
})

model = XGBClassifier(**best)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)
print(f"Best iteration : {model.best_iteration}")

[0]	validation_0-aucpr:0.86848	validation_1-aucpr:0.87911
[50]	validation_0-aucpr:0.99994	validation_1-aucpr:0.99986
[100]	validation_0-aucpr:0.99999	validation_1-aucpr:0.99990
[112]	validation_0-aucpr:1.00000	validation_1-aucpr:0.99989
Best iteration : 92


---

### Evaluate

In [20]:
y_prob = model.predict_proba(X_test)[:, 1]

In [21]:
from sklearn.metrics import f1_score, precision_score, recall_score

auc_roc = roc_auc_score(y_test, y_prob)
auc_pr  = average_precision_score(y_test, y_prob)

# Chọn threshold tối đa hóa F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = f1s.argmax()
best_threshold = float(thresholds[best_idx])

y_pred = (y_prob >= best_threshold).astype(int)
p  = precision_score(y_test, y_pred)
r  = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Threshold : {best_threshold:.4f}\n")
print(f"Precision : {(p):.2f}  {'OK' if p > 0.80 else 'FAIL'}  (target > 0.80)")
print(f"Recall    : {(r):.2f}  {'OK' if r > 0.75 else 'FAIL'}  (target > 0.75)")
print(f"F1-Score  : {(f1):.2f}  {'OK' if f1 > 0.78 else 'FAIL'}  (target > 0.78)")
print(f"AUC-ROC   : {(auc_roc):.2f}  {'OK' if auc_roc > 0.85 else 'FAIL'}  (target > 0.85)")
print(f"AUC-PR    : {(auc_pr):.2f}  {'OK' if auc_pr > 0.70 else 'FAIL'}  (target > 0.70)")


Threshold : 0.9184

Precision : 1.00  OK  (target > 0.80)
Recall    : 1.00  OK  (target > 0.75)
F1-Score  : 1.00  OK  (target > 0.78)
AUC-ROC   : 1.00  OK  (target > 0.85)
AUC-PR    : 1.00  OK  (target > 0.70)


---

### Save Model

In [22]:
out_path = Path("../data/model_detection.json")
out_path.parent.mkdir(exist_ok=True)

model.get_booster().save_model(str(out_path))

print(f"Saved : {out_path}")
print(f"Size  : {out_path.stat().st_size / 1024:.1f} KB")
print(f"Threshold dùng trong Flink: {best_threshold:.4f}")

Saved : ../data/model_detection.json
Size  : 247.5 KB
Threshold dùng trong Flink: 0.9184
